In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.cluster import KMeans
import lightgbm as lgbm

In [3]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("muted")
plt.rcParams['figure.figsize'] = (12, 8)

In [5]:
def load_data(file_path, is_train=True):
    """
    Load the data from CMAPSS dataset files
    """
    # Define column names
    columns = ['unit_number', 'time_in_cycles', 'setting_1', 'setting_2', 'setting_3',
               'T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr',
               'Ps30', 'phi', 'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd',
               'PCNfR_dmd', 'W31', 'W32']
    
    # Read the data
    data = pd.read_csv(file_path, sep=' ', header=None, names=columns)
    
    # Remove NaN values (from the double spaces in the text file)
    data.dropna(axis=1, inplace=True)
    
    if is_train:
        # For training data - calculate RUL
        # Get the maximum cycle for each unit
        max_cycles = data.groupby('unit_number')['time_in_cycles'].max().reset_index()
        max_cycles.columns = ['unit_number', 'max_cycles']
        
        # Merge with the original data
        data = data.merge(max_cycles, on=['unit_number'], how='left')
        
        # Calculate RUL: max_cycles - current_cycle
        data['RUL'] = data['max_cycles'] - data['time_in_cycles']
        
        # Drop the max_cycles column
        data.drop('max_cycles', axis=1, inplace=True)
    
    return data


In [6]:
def load_test_data_with_rul(test_file_path, rul_file_path):
    """
    Load test data and combine with the RUL values
    """
    # Load test data
    test_data = load_data(test_file_path, is_train=False)
    
    # Load RUL values
    rul_values = pd.read_csv(rul_file_path, sep=' ', header=None)
    rul_values.dropna(axis=1, inplace=True)
    rul_values.columns = ['RUL']
    
    # Get the latest cycle data for each unit
    latest_cycles = test_data.groupby('unit_number').tail(1).reset_index(drop=True)
    
    # Add the RUL values to the latest cycle data
    latest_cycles['RUL'] = rul_values['RUL'].values
    
    return test_data, latest_cycles


In [7]:
# Function to perform feature engineering and selection
def feature_engineering(data):
    """
    Create new features and select the most important ones
    """
    # Create new features
    # Operating conditions (combination of settings)
    data['op_setting'] = data['setting_1'] * data['setting_2'] / data['setting_3']
    
    # Temperature ratio features
    data['T_ratio'] = data['T24'] / data['T30']
    data['T_diff'] = data['T30'] - data['T24']
    
    # Pressure ratio features
    data['P_ratio'] = data['P30'] / data['P2']
    
    # Vibration features (assuming higher values in certain sensors indicate more wear)
    data['sensor_ratio_1'] = data['T50'] / data['T30']
    
    # Additional engineered features for Ridge and LightGBM
    data['P_T_interaction'] = data['P30'] * data['T30']
    data['normalized_cycle'] = data['time_in_cycles'] / data.groupby('unit_number')['time_in_cycles'].transform('max')
    data['setting_total'] = data['setting_1'] + data['setting_2'] + data['setting_3']
    data['T_sum'] = data['T2'] + data['T24'] + data['T30'] + data['T50']
    data['P_sum'] = data['P2'] + data['P15'] + data['P30']
    
    # More advanced features can be added based on domain knowledge
    
    return data

In [8]:
def visualize_data(train_data, test_data=None):
    """
    Create visualizations for data exploration
    """
    # 1. Distribution of RUL in training data
    plt.figure(figsize=(10, 6))
    sns.histplot(train_data['RUL'], kde=True)
    plt.title('Distribution of Remaining Useful Life (RUL)')
    plt.xlabel('Remaining Useful Life (cycles)')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.savefig('rul_distribution.png')
    plt.close()
    
    # 2. Sensor readings over time for a sample unit
    unit_sample = train_data[train_data['unit_number'] == 1]
    
    plt.figure(figsize=(15, 10))
    sensor_cols = ['T24', 'T30', 'T50', 'P30', 'Nc', 'NRc']
    for i, sensor in enumerate(sensor_cols, 1):
        plt.subplot(2, 3, i)
        plt.plot(unit_sample['time_in_cycles'], unit_sample[sensor])
        plt.title(f'Sensor {sensor} vs Cycles')
        plt.xlabel('Cycles')
        plt.ylabel(sensor)
    plt.tight_layout()
    plt.savefig('sensor_vs_cycles.png')
    plt.close()
    
    # 3. Correlation heatmap
    plt.figure(figsize=(16, 14))
    selected_cols = ['setting_1', 'setting_2', 'setting_3', 'T24', 'T30', 'T50',
                    'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 'NRf',
                    'NRc', 'BPR', 'RUL']
    correlation = train_data[selected_cols].corr()
    mask = np.triu(np.ones_like(correlation, dtype=bool))
    sns.heatmap(correlation, mask=mask, annot=False, cmap='coolwarm', center=0)
    plt.title('Correlation Heatmap of Sensor Readings and RUL')
    plt.tight_layout()
    plt.savefig('correlation_heatmap.png')
    plt.close()
    
    # 4. Scatter plot of time vs RUL for different units
    plt.figure(figsize=(12, 8))
    sample_units = train_data[train_data['unit_number'].isin([1, 2, 3, 4, 5])]
    for unit, data in sample_units.groupby('unit_number'):
        plt.scatter(data['time_in_cycles'], data['RUL'], label=f'Unit {unit}')
    plt.xlabel('Time in Cycles')
    plt.ylabel('RUL')
    plt.title('RUL vs Time for Different Units')
    plt.legend()
    plt.tight_layout()
    plt.savefig('rul_vs_time.png')
    plt.close()
    
    # 5. Distribution plot comparing an important sensor across different RUL ranges
    plt.figure(figsize=(12, 8))
    train_data['RUL_bin'] = pd.cut(train_data['RUL'], bins=[0, 50, 100, 150, 200, 250, 300], 
                                   labels=['0-50', '50-100', '100-150', '150-200', '200-250', '250-300'])
    sns.violinplot(x='RUL_bin', y='T30', data=train_data)
    plt.title('Distribution of T30 Across Different RUL Ranges')
    plt.xlabel('RUL Range')
    plt.ylabel('T30 Value')
    plt.tight_layout()
    plt.savefig('sensor_distribution_by_rul.png')
    plt.close()
    
    # 6. Compare treatment effect visualization (similar to the poster)
    if test_data is not None:
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        sns.kdeplot(train_data['T30'], label='Train')
        sns.kdeplot(test_data['T30'], label='Test')
        plt.title('Density Plot of T30')
        plt.legend()
        
        plt.subplot(1, 2, 2)
        sns.kdeplot(train_data['Nc'], label='Train')
        sns.kdeplot(test_data['Nc'], label='Test')
        plt.title('Density Plot of Nc')
        plt.legend()
        plt.tight_layout()
        plt.savefig('train_test_comparison.png')
        plt.close()


In [9]:
def prepare_for_modeling(data, is_train=True, scaler=None):
    """
    Prepare the data for modeling by creating features and scaling
    """
    # Feature engineering
    data = feature_engineering(data)
    
    # Select features for modeling
    feature_cols = ['setting_1', 'setting_2', 'setting_3', 'T2', 'T24', 'T30', 'T50',
                    'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 'NRf',
                    'NRc', 'BPR', 'op_setting', 'T_ratio', 'T_diff', 'P_ratio', 
                    'sensor_ratio_1', 'P_T_interaction', 'normalized_cycle', 
                    'setting_total', 'T_sum', 'P_sum']
    
    if is_train:
        X = data[feature_cols]
        y = data['RUL']
        
        # Scale the features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
        
        return X_scaled, y, scaler
    else:
        X = data[feature_cols]
        
        # Scale the features using the provided scaler
        X_scaled = scaler.transform(X)
        X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
        
        return X_scaled


In [10]:
def train_models(X_train, y_train, X_val=None, y_val=None):
    """
    Train different models and evaluate their performance
    """
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge Regression': Ridge(alpha=10.0),  # Added Ridge Regression
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
        'LightGBM': lgbm.LGBMRegressor(  # Added LightGBM
            objective='regression',
            num_leaves=31,
            learning_rate=0.05,
            n_estimators=100,
            random_state=42
        )
    }
    
    results = {}
    trained_models = {}
    
    for name, model in models.items():
        print(f"Training {name}...")
        model.fit(X_train, y_train)
        trained_models[name] = model
        
        # Make predictions
        train_preds = model.predict(X_train)
        
        # Evaluate on training set
        train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
        train_mae = mean_absolute_error(y_train, train_preds)
        train_r2 = r2_score(y_train, train_preds)
        
        results[name] = {
            'train_rmse': train_rmse,
            'train_mae': train_mae,
            'train_r2': train_r2
        }
        
        # If validation set is provided, evaluate on it
        if X_val is not None and y_val is not None:
            val_preds = model.predict(X_val)
            val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
            val_mae = mean_absolute_error(y_val, val_preds)
            val_r2 = r2_score(y_val, val_preds)
            
            results[name]['val_rmse'] = val_rmse
            results[name]['val_mae'] = val_mae
            results[name]['val_r2'] = val_r2
    
    return trained_models, results

In [11]:
def tune_hyperparameters(X_train, y_train):
    """
    Perform hyperparameter tuning for Ridge Regression and LightGBM
    """
    print("Tuning Ridge Regression hyperparameters...")
    ridge_params = {'alpha': [0.1, 1.0, 10.0, 50.0, 100.0]}
    ridge_grid = GridSearchCV(Ridge(), ridge_params, cv=5, scoring='neg_mean_squared_error')
    ridge_grid.fit(X_train, y_train)
    best_ridge_alpha = ridge_grid.best_params_['alpha']
    print(f"Best Ridge alpha: {best_ridge_alpha}")
    
    print("Tuning LightGBM hyperparameters...")
    lgbm_params = {
        'num_leaves': [31, 50, 70],
        'learning_rate': [0.01, 0.05, 0.1],
        'n_estimators': [50, 100, 200]
    }
    lgbm_grid = GridSearchCV(lgbm.LGBMRegressor(random_state=42), lgbm_params, cv=5, scoring='neg_mean_squared_error')
    lgbm_grid.fit(X_train, y_train)
    best_lgbm_params = lgbm_grid.best_params_
    print(f"Best LightGBM parameters: {best_lgbm_params}")
    
    best_models = {
        'Ridge Regression': Ridge(alpha=best_ridge_alpha),
        'LightGBM': lgbm.LGBMRegressor(
            objective='regression',
            num_leaves=best_lgbm_params['num_leaves'],
            learning_rate=best_lgbm_params['learning_rate'],
            n_estimators=best_lgbm_params['n_estimators'],
            random_state=42
        )
    }
    
    return best_models

In [12]:
def visualize_model_performance(results):
    """
    Visualize the performance of different models
    """
    # Convert results to DataFrame for easier plotting
    metrics = ['train_rmse', 'train_mae', 'train_r2']
    if 'val_rmse' in results['Linear Regression']:
        metrics.extend(['val_rmse', 'val_mae', 'val_r2'])
    
    data = []
    for model_name, model_results in results.items():
        for metric in metrics:
            data.append({
                'Model': model_name,
                'Metric': metric,
                'Value': model_results[metric]
            })
    
    df_results = pd.DataFrame(data)
    
    # Plot the results
    plt.figure(figsize=(15, 8))
    
    # RMSE
    plt.subplot(1, 3, 1)
    sns.barplot(x='Model', y='Value', data=df_results[df_results['Metric'].isin(['train_rmse', 'val_rmse'])])
    plt.title('RMSE by Model')
    plt.xticks(rotation=45)
    plt.legend(['Train', 'Validation'] if 'val_rmse' in df_results['Metric'].values else ['Train'])
    
    # MAE
    plt.subplot(1, 3, 2)
    sns.barplot(x='Model', y='Value', data=df_results[df_results['Metric'].isin(['train_mae', 'val_mae'])])
    plt.title('MAE by Model')
    plt.xticks(rotation=45)
    plt.legend(['Train', 'Validation'] if 'val_mae' in df_results['Metric'].values else ['Train'])
    
    # R2
    plt.subplot(1, 3, 3)
    sns.barplot(x='Model', y='Value', data=df_results[df_results['Metric'].isin(['train_r2', 'val_r2'])])
    plt.title('R² by Model')
    plt.xticks(rotation=45)
    plt.legend(['Train', 'Validation'] if 'val_r2' in df_results['Metric'].values else ['Train'])
    
    plt.tight_layout()
    plt.savefig('model_performance.png')
    plt.close()

# Function to analyze feature importance
def analyze_feature_importance(models, feature_names):
    """
    Analyze and visualize feature importance from trained models
    """
    feature_importance_data = {}
    
    # Get feature importance from Random Forest model
    if 'Random Forest' in models:
        rf_model = models['Random Forest']
        rf_importance = rf_model.feature_importances_
        feature_importance_data['Random Forest'] = rf_importance
    
    # Get feature importance from LightGBM
    if 'LightGBM' in models:
        lgbm_model = models['LightGBM']
        lgbm_importance = lgbm_model.feature_importances_
        feature_importance_data['LightGBM'] = lgbm_importance
    
    # Get coefficients from Ridge Regression (absolute values)
    if 'Ridge Regression' in models:
        ridge_model = models['Ridge Regression']
        ridge_importance = np.abs(ridge_model.coef_)
        feature_importance_data['Ridge Regression'] = ridge_importance
    
    # Plot feature importance for each model
    for model_name, importance in feature_importance_data.items():
        # Create a DataFrame for feature importance
        feat_imp = pd.DataFrame({
            'Feature': feature_names,
            'Importance': importance
        }).sort_values('Importance', ascending=False)
        
        # Plot the feature importance
        plt.figure(figsize=(12, 8))
        sns.barplot(x='Importance', y='Feature', data=feat_imp.head(15))
        plt.title(f'Feature Importance ({model_name})')
        plt.tight_layout()
        plt.savefig(f'feature_importance_{model_name.replace(" ", "_").lower()}.png')
        plt.close()
    
    # Return the Random Forest importance by default, or the first available one
    if feature_importance_data:
        first_model = list(feature_importance_data.keys())[0]
        first_importance = feature_importance_data[first_model]
        
        feat_imp = pd.DataFrame({
            'Feature': feature_names,
            'Importance': first_importance
        }).sort_values('Importance', ascending=False)
        
        return feat_imp
    
    return None

In [13]:
def predict_rul(models, X_test, best_model_name='LightGBM'):
    """
    Make predictions on test data using the best model
    """
    best_model = models[best_model_name]
    predictions = best_model.predict(X_test)
    
    return predictions

In [14]:
def visualize_predictions(y_true, y_pred, title='Predictions vs Actual'):
    """
    Visualize predictions vs actual values
    """
    plt.figure(figsize=(12, 8))
    plt.scatter(y_true, y_pred, alpha=0.5)
    
    # Add the identity line
    min_val = min(min(y_true), min(y_pred))
    max_val = max(max(y_true), max(y_pred))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')
    
    plt.title(title)
    plt.xlabel('Actual RUL')
    plt.ylabel('Predicted RUL')
    plt.tight_layout()
    plt.savefig('predictions_vs_actual.png')
    plt.close()

In [15]:
def compare_model_predictions(models, X_test, y_true):
    """
    Compare predictions from different models
    """
    predictions = {}
    metrics = {}
    
    for name, model in models.items():
        preds = model.predict(X_test)
        predictions[name] = preds
        
        rmse = np.sqrt(mean_squared_error(y_true, preds))
        mae = mean_absolute_error(y_true, preds)
        r2 = r2_score(y_true, preds)
        
        metrics[name] = {
            'RMSE': rmse,
            'MAE': mae,
            'R²': r2
        }
    
    # Create a DataFrame for model comparison
    metrics_df = pd.DataFrame(metrics).T
    
    # Plot the comparison
    plt.figure(figsize=(15, 10))
    
    # Plot actual vs predicted for each model
    colors = ['blue', 'red', 'green', 'purple', 'orange']
    for i, (name, preds) in enumerate(predictions.items()):
        plt.subplot(2, 3, i + 1)
        plt.scatter(y_true, preds, alpha=0.5, color=colors[i % len(colors)])
        
        # Add the identity line
        min_val = min(min(y_true), min(preds))
        max_val = max(max(y_true), max(preds))
        plt.plot([min_val, max_val], [min_val, max_val], 'r--')
        
        plt.title(f'{name}\nRMSE: {metrics[name]["RMSE"]:.2f}, R²: {metrics[name]["R²"]:.2f}')
        plt.xlabel('Actual RUL')
        plt.ylabel('Predicted RUL')
    
    plt.tight_layout()
    plt.savefig('model_predictions_comparison.png')
    plt.close()
    
    return metrics_df

In [16]:
def main():
    # Load and prepare the train data
    print("Loading and preparing data...")
    train_data = load_data('train_FD001.txt')
    
    # Load test data and RUL values
    test_data, test_with_rul = load_test_data_with_rul('test_FD001.txt', 'RUL_FD001.txt')
    
    # Visualize the data
    print("Visualizing data...")
    visualize_data(train_data, test_data)
    
    # Prepare data for modeling
    print("Preparing data for modeling...")
    X_train, y_train, scaler = prepare_for_modeling(train_data)
    
    # Split train data into train and validation sets
    X_train_split, X_val, y_train_split, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42
    )
    
    # Hyperparameter tuning for Ridge and LightGBM
    print("Tuning hyperparameters for Ridge Regression and LightGBM...")
    tuned_models = tune_hyperparameters(X_train_split, y_train_split)
    
    # Train and evaluate models
    print("Training base models...")
    trained_models, results = train_models(X_train_split, y_train_split, X_val, y_val)
    
    # Train tuned models
    print("Training tuned models...")
    for name, model in tuned_models.items():
        print(f"Training tuned {name}...")
        model.fit(X_train_split, y_train_split)
        trained_models[f"Tuned {name}"] = model
        
        # Make predictions
        train_preds = model.predict(X_train_split)
        val_preds = model.predict(X_val)
        
        # Evaluate
        train_rmse = np.sqrt(mean_squared_error(y_train_split, train_preds))
        train_mae = mean_absolute_error(y_train_split, train_preds)
        train_r2 = r2_score(y_train_split, train_preds)
        
        val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
        val_mae = mean_absolute_error(y_val, val_preds)
        val_r2 = r2_score(y_val, val_preds)
        
        results[f"Tuned {name}"] = {
            'train_rmse': train_rmse,
            'train_mae': train_mae,
            'train_r2': train_r2,
            'val_rmse': val_rmse,
            'val_mae': val_mae,
            'val_r2': val_r2
        }
    
    # Visualize model performance
    print("Visualizing model performance...")
    visualize_model_performance(results)
    
    # Analyze feature importance
    print("Analyzing feature importance...")
    feat_imp = analyze_feature_importance(trained_models, X_train.columns)
    if feat_imp is not None:
        print("Top 10 Important Features:")
        print(feat_imp.head(10))
    
    # Prepare test data for prediction
    X_test = prepare_for_modeling(test_with_rul, is_train=False, scaler=scaler)
    
    # Compare model predictions
    print("Comparing model predictions...")
    metrics_df = compare_model_predictions(trained_models, X_test, test_with_rul['RUL'])
    print("Model Comparison Metrics:")
    print(metrics_df)
    
    # Determine the best model based on validation RMSE
    best_model_name = min(results, key=lambda x: results[x].get('val_rmse', float('inf')))
    print(f"Best model based on validation RMSE: {best_model_name}")
    
    # Make predictions on test data using the best model
    print(f"Making predictions on test data using {best_model_name}...")
    y_pred = predict_rul(trained_models, X_test, best_model_name)
    
    # Visualize predictions vs actual
    print("Visualizing predictions...")
    visualize_predictions(test_with_rul['RUL'], y_pred, f'Predicted vs Actual RUL ({best_model_name})')
    
    # Calculate metrics for test predictions
    rmse = np.sqrt(mean_squared_error(test_with_rul['RUL'], y_pred))
    mae = mean_absolute_error(test_with_rul['RUL'], y_pred)
    r2 = r2_score(test_with_rul['RUL'], y_pred)
    
    print(f"Test RMSE: {rmse:.4f}")
    print(f"Test MAE: {mae:.4f}")
    print(f"Test R²: {r2:.4f}")
    
    # Save predictions to a file
    results_df = pd.DataFrame({
        'unit_number': test_with_rul['unit_number'],
        'actual_RUL': test_with_rul['RUL'],
        'predicted_RUL': y_pred
    })
    
    results_df.to_csv('predicted_rul_results.csv', index=False)
    print("Results saved to 'predicted_rul_results.csv'")

if __name__ == "__main__":
    main()

Loading and preparing data...


FileNotFoundError: [Errno 2] No such file or directory: 'train_FD001.txt'